# LEAA Training — Stage 3: `static_far`
**Target accuracy:** 85%  |  **Timesteps:** 15,000,000

### Setup Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Add secrets in the left sidebar (🔑 Secrets):
   - `GITHUB_TOKEN` — your GitHub Personal Access Token
   - `GMAIL_ADDRESS` — your Gmail address *(optional, for notifications)*
   - `GMAIL_APP_PASSWORD` — Gmail App Password *(optional)*
     → [Create App Password](https://myaccount.google.com/apppasswords) (requires 2FA enabled)
3. Run all cells in order
4. When session expires, re-open and run all cells — training auto-resumes


In [ ]:
# Cell SSH: SSH into this Colab VM + copy .env credentials
# Run this cell FIRST — before Cell 0
# Requires cloudflared on your Mac: brew install cloudflared
import os, subprocess, threading, time, random, string

SSH_PASSWORD = ''.join(random.choices(string.ascii_letters + string.digits, k=12))

os.system('apt-get install -qq openssh-server > /dev/null 2>&1')
os.system(f'echo "root:{SSH_PASSWORD}" | chpasswd')
os.system('mkdir -p /run/sshd')
os.system('echo "PermitRootLogin yes" >> /etc/ssh/sshd_config')
os.system('echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config')
os.system('/usr/sbin/sshd')

os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
os.system('chmod +x /usr/local/bin/cloudflared')

tunnel_url = [None]

def run_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22'],
        stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
    )
    for line in proc.stderr:
        if 'trycloudflare.com' in line:
            import re
            match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if match:
                tunnel_url[0] = match.group(0)
                break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()
for _ in range(30):
    if tunnel_url[0]: break
    time.sleep(1)

if tunnel_url[0]:
    h = tunnel_url[0].replace('https://', '')
    print('\n✓ SSH tunnel active!')
    print(f'  Password : {SSH_PASSWORD}')
    print(f'\n  SSH connect:')
    print(f'  ssh -o ProxyCommand="cloudflared access ssh --hostname {h}" root@{h}')
    print(f'\n  Copy .env to VM (run in your Mac terminal BEFORE running Cell 0):')
    print(f'  sshpass -p "{SSH_PASSWORD}" scp -o ProxyCommand="cloudflared access ssh --hostname {h}" -o StrictHostKeyChecking=no /path/to/LEAA/.env root@{h}:/tmp/.env')
    print(f'  sshpass -p "{SSH_PASSWORD}" ssh -o ProxyCommand="cloudflared access ssh --hostname {h}" -o StrictHostKeyChecking=no root@{h} "mkdir -p /content/leaa && cp /tmp/.env /content/leaa/.env"')
else:
    print('✗ Tunnel failed to start')


In [ ]:
# Cell 0: Load credentials from .env file
# Copy your local .env to the VM using the scp command printed by the SSH cell
import os
from dotenv import load_dotenv

load_dotenv('/content/leaa/.env')

required = ['GITHUB_TOKEN']
for key in required:
    if not os.environ.get(key):
        raise RuntimeError(f'{key} not set — copy your .env file to the VM first')

print(f'✓ Credentials loaded from .env')


In [ ]:
# Cell 2: Authenticate & clone repo
import os, subprocess

TOKEN = os.environ.get('GITHUB_TOKEN')
if not TOKEN:
    raise RuntimeError('GITHUB_TOKEN not set — run Cell 0 first')
REPO = 'Sathvik-Chowdary-Veerapaneni/Language-Embeded-Agent-Action'
CLONE_URL = f'https://{TOKEN}@github.com/{REPO}.git'

if not os.path.exists('/content/leaa'):
    subprocess.run(['git', 'clone', CLONE_URL, '/content/leaa'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/leaa', check=True)

subprocess.run(['git', 'config', 'user.email', 'colab@leaa.bot'], cwd='/content/leaa')
subprocess.run(['git', 'config', 'user.name', 'Colab Training Bot'], cwd='/content/leaa')
subprocess.run(['git', 'remote', 'set-url', 'origin', CLONE_URL], cwd='/content/leaa')
print('✓ Repo ready at /content/leaa')


In [ ]:
# Cell 3: Load email credentials (optional)
import os

GMAIL_ADDRESS      = os.environ.get('GMAIL_ADDRESS')
GMAIL_APP_PASSWORD = os.environ.get('GMAIL_APP_PASSWORD')

if GMAIL_ADDRESS and GMAIL_APP_PASSWORD:
    print(f'✓ Email notifications enabled → {GMAIL_ADDRESS}')
else:
    print('⚠ Email notifications disabled')


In [ ]:
# Cell 4: Install dependencies
%cd /content/leaa
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

In [ ]:
# Cell 5: Run training
# Runs for up to 23h. Checkpoints sync to GitHub every 5 min.
# Runtime watchdog emails a warning at 22h and stops training at 23h
# so the VM has 1h to finish saving before Colab reclaims it.
# If the session expires, re-run all cells — training resumes from last checkpoint.
%cd /content/leaa
import os

cmd = 'python scripts/colab_train.py --stage 3 --timesteps 15000000 --num-envs 8 --max-runtime-hours 23 --sync-interval 300'

# Append email args if credentials are available
if 'GMAIL_ADDRESS' in dir() and GMAIL_ADDRESS:
    cmd += f' --gmail {GMAIL_ADDRESS} --gmail-password {GMAIL_APP_PASSWORD}'

print(f'Running: {cmd}')
os.system(cmd)

In [ ]:
# Cell 6: (Optional) Evaluate this stage after training
%cd /content/leaa
!python rl_training/evaluate.py \\
    --model rl_training/checkpoints/static_far_best.zip \\
    --vecnorm rl_training/checkpoints/vecnormalize_static_far_best.pkl \\
    --stage static_far \\
    --episodes 200